# AI Meeting Assistant
Turn meeting transcript text into draft summaries, decisions, action items, and follow-up emails.

**Safety:** Obtain consent before processing meetings. Review all generated minutes before sharing. Do not send confidential transcripts to external models unless authorized.

## 1. Setup and sample transcript

In [ ]:
import os, re, json
from pathlib import Path
from datetime import date

meeting_title = "Product Launch Readiness"
meeting_date = str(date.today())
transcript = '''
Alex: Thanks everyone. Today we need to finalize the launch checklist.
Priya: The API integration is complete. I will run regression tests by Thursday.
Jordan: We agreed to move the launch review to Friday at 10 AM.
Alex: I will update the project timeline and share it tomorrow.
Priya: The open issue is payment retry behavior. I will document the edge cases.
Jordan: I will confirm support team coverage before Friday.
Alex: Decision: the release candidate must pass regression tests before approval.
'''
print("Transcript loaded:", len(transcript.split()), "words")

## 2. Clean transcript and identify speaker turns

In [ ]:
def clean_transcript(text):
    text = text.replace("\\r", "\\n")
    text = re.sub(r"[ \\t]+", " ", text)
    return re.sub(r"\\n{3,}", "\\n\\n", text).strip()

def speaker_turns(text):
    turns = []
    pattern = re.compile(r"^\\s*([A-Za-z][A-Za-z0-9 _.-]{0,35}):\\s*(.+)$")
    for line in clean_transcript(text).splitlines():
        m = pattern.match(line)
        if m:
            turns.append({"speaker": m.group(1).strip(), "text": m.group(2).strip()})
        elif line.strip() and turns:
            turns[-1]["text"] += " " + line.strip()
    return turns

cleaned = clean_transcript(transcript)
turns = speaker_turns(cleaned)
for t in turns:
    print(f"{t['speaker']}: {t['text']}")

## 3. Generate a baseline summary

In [ ]:
def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\\s+", text) if s.strip()]

def summarize(text, limit=5):
    terms = {"decision", "agreed", "action", "deadline", "launch", "issue", "will", "complete"}
    ranked = []
    for i, sentence in enumerate(sentences(text)):
        words = re.findall(r"[a-z]+", sentence.lower())
        score = sum(w in terms for w in words) + min(len(words) / 20, 1)
        ranked.append((score, i, sentence))
    return [x[2] for x in sorted(sorted(ranked, reverse=True)[:limit], key=lambda x: x[1])]

summary = summarize(cleaned)
summary

## 4. Extract candidate decisions and action items

In [ ]:
action_pattern = re.compile(r"\\b(?:i will|we will|please|action item)\\b", re.I)
decision_pattern = re.compile(r"\\b(?:decision:|we agreed|agreed to|decided to)\\b", re.I)

actions, decisions = [], []
for turn in turns:
    text = turn["text"]
    if action_pattern.search(text):
        actions.append({"owner": turn["speaker"], "task": text,
                        "due_date": None, "status": "Needs confirmation"})
    if decision_pattern.search(text):
        decisions.append(text)

print("Candidate action items:", json.dumps(actions, indent=2))
print("Candidate decisions:", json.dumps(decisions, indent=2))

## 5. Optional LLM enhancement
Configure environment variables before running. This step is skipped without an API key. Only use an approved endpoint for confidential information.

In [ ]:
def optional_llm_minutes(text):
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        print("No API key configured; use baseline output.")
        return None
    from openai import OpenAI  # optional: pip install openai
    kwargs = {"api_key": key}
    if os.getenv("OPENAI_BASE_URL"):
        kwargs["base_url"] = os.getenv("OPENAI_BASE_URL")
    client = OpenAI(**kwargs)
    prompt = f\'''Create accurate meeting minutes using only the transcript.
Return JSON with keys: summary, decisions, action_items (owner, task, due_date, status),
risks_or_open_questions, follow_up_email. Do not invent facts; mark uncertainty.

TRANSCRIPT:
{text}
\'''
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
        temperature=0.2,
        messages=[{"role": "system", "content": "You create faithful meeting minutes. Output JSON only."},
                  {"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)

llm_result = optional_llm_minutes(cleaned)

## 6. Assemble minutes and export JSON/Markdown

In [ ]:
if llm_result:
    minutes = {"meeting_title": meeting_title, "meeting_date": meeting_date, **llm_result}
else:
    minutes = {
        "meeting_title": meeting_title, "meeting_date": meeting_date,
        "summary": summary, "decisions": decisions, "action_items": actions,
        "risks_or_open_questions": ["Confirm whether other risks or questions were omitted."],
        "follow_up_email": (
            f"Subject: Follow-up — {meeting_title}\\n\\n"
            "Hi everyone,\\n\\nThank you for the discussion. Please review the draft "
            "minutes and action items and send any corrections.\\n\\nBest,\\nMeeting Organizer"
        )
    }

def to_markdown(data):
    out = [f"# {data['meeting_title']}", f"**Date:** {data['meeting_date']}", "",
           "## Summary"]
    out += [f"- {x}" for x in data.get("summary", [])]
    out += ["", "## Decisions"] + [f"- {x}" for x in data.get("decisions", [])]
    out += ["", "## Action items"]
    for item in data.get("action_items", []):
        out.append(f"- [ ] **{item.get('owner') or 'Owner TBD'}** — {item.get('task','')} "
                   f"(Due: {item.get('due_date') or 'TBD'})")
    out += ["", "## Risks / open questions"]
    out += [f"- {x}" for x in data.get("risks_or_open_questions", [])]
    out += ["", "## Follow-up email", "", data.get("follow_up_email", "")]
    return "\\n".join(out)

outdir = Path("meeting_assistant_output")
outdir.mkdir(exist_ok=True)
(outdir / "meeting_minutes.json").write_text(json.dumps(minutes, indent=2), encoding="utf-8")
(outdir / "meeting_minutes.md").write_text(to_markdown(minutes), encoding="utf-8")
print("Exported meeting minutes to", outdir)

## 7. Human review checklist
- Verify summaries against the transcript.
- Confirm each decision was approved rather than merely discussed.
- Confirm action owners, tasks, and deadlines.
- Check names, dates, numbers, and acronyms.
- Follow consent, privacy, retention, and workplace policies.

## Limitations
This starter accepts transcript text; it does not record meetings or transcribe audio. Speaker extraction requires `Speaker: text` lines. Heuristic and LLM outputs can be wrong or incomplete. Human review is required before distribution.